### Import Libraries


In [139]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd
from TextPreprocessor import TextPreprocessor
from sklearn.model_selection import train_test_split
import json
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

### Preprocess Datasets
saves a csv with stop words and splits the data


In [140]:
dataset = pd.read_csv("../data/imdb_dataset.csv")
preprocessor = TextPreprocessor(True, True, True)

dataset = preprocessor.process(dataset, textcol="review", drop_original=True)


if dataset['sentiment'].dtype == 'object':
    dataset['sentiment'] = dataset['sentiment'].map({'positive': 1, 'negative': 0})

X_train, X_test, y_train, y_test = train_test_split(
    dataset["cleaned_review"].values, dataset["sentiment"].values,
    test_size=0.2, random_state=42, stratify=dataset["sentiment"].values
)
dataset.to_csv("../data/imdb_cleaned_with_stop.csv", index=False)

### Set Tokenizer Parameters


In [141]:
vocab_size = 20000
max_length = 300
oov_token = "<OOV>"

### Tokenizes and Saves it 


In [142]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_padded = pad_sequences(X_train_sequences, maxlen=max_length, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_length, padding='post', truncating='post')

tok_json = tokenizer.to_json()
with open('../models/tokenizer.json', 'w') as json_file:
    json_file.write(tok_json)

    

### Import Keras Models and Layers


In [143]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Bidirectional
from tensorflow.keras import regularizers

### MLP Model

In [144]:
mlp_model = Sequential([
        Embedding(vocab_size, 64, input_length=max_length),
        GlobalAveragePooling1D(),
        Dense(64, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])

/home/jason/ann/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### LTSM Model

In [ ]:
ltsm_model = Sequential([
        Embedding(vocab_size, 128, input_length=max_length),
        Bidirectional(LSTM(128)),
        
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])

NameError: name 'l2' is not defined

### CNN Model

In [ ]:
cnn_model = Sequential([
        Embedding(vocab_size, 128, input_length=max_length),
        Conv1D(128, kernel_size=5, activation="relu"),
        MaxPooling1D(pool_size=2),
        Conv1D(128, kernel_size=5, activation="relu"),
        GlobalAveragePooling1D(),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])

### Compile the Models and Summary

In [ ]:
mlp_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
ltsm_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
cnn_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
mlp_model.summary()
ltsm_model.summary()
cnn_model.summary()

### Train with Callbacks 
early stoppping and best checkpoint

In [ ]:
ckpt_path = "../models/nn_models/best.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True)
]

### Train MLP 

In [ ]:
mlp_history = mlp_model.fit(
    X_train_padded, y_train, 
    validation_data=(X_test_padded, y_test), 
    epochs=10, 
    batch_size=32, 
    callbacks=callbacks, 
    verbose=1
    )

### Train LTSM


In [ ]:
ltsm_history = ltsm_model.fit(
    X_train_padded, y_train, 
    validation_data=(X_test_padded, y_test), 
    epochs=10, 
    batch_size=32, 
    callbacks=callbacks, 
    verbose=1
    )

### Train CNN

In [ ]:
cnn_history = cnn_model.fit(
    X_train_padded, y_train, 
    validation_data=(X_test_padded, y_test), 
    epochs=10, 
    batch_size=32, 
    callbacks=callbacks, 
    verbose=1
    )

In [ ]:
y_prob = mlp_model.predict(X_test_padded).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, digits=3))


cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix MLP")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()


In [ ]:
y_prob = ltsm_model.predict(X_test_padded).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, digits=3))


cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix LTSM")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()


In [ ]:
y_prob = cnn_model.predict(X_test_padded).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, digits=3))


cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix CNN")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()


In [ ]:
def predict_review(text: str): 
    ptext = preprocessor.clean(text)
    seq = tokenizer.texts_to_sequences([ptext])
    pad = pad_sequences(seq, maxlen=max_length, padding="post", truncating="post")
    p = float(cnn_model.predict(pad)[0][0])
    label = "Positive" if p>=0.5 else "Negative"
    return label, p

predict_review("I THINK THIS MOVIE WAS AMAZING")
